In [22]:
import re
import json

In [124]:
class BPEMethod:
    def __init__(self, file_path: str):
        with open(file_path) as file:
            self.text = self.format_text(file.read())
            self.tokens = list(self.text)
            self.codes = {}

    def format_text(self, text: str):
        text = text.lower()
        text = re.sub(r'[^а-яёa-z0-9 ,.!?-]', '', text)
        text = re.sub(r' +', ' ', text)

        return text

    def get_counts(self):
        counts = {}
        for a, b in zip(self.tokens, self.tokens[1:]):
            key = (a, b)
            counts[key] = counts.get(key, 0) + 1
        return counts
            
    def check_bytes(self, codes: dict):
        serializable_codes = {str(k): list(v) for k, v in codes.items()}
        json_string = json.dumps(serializable_codes, separators=(',', ':'), ensure_ascii=False)
        return len(json_string.encode('utf-8')) <= 1024

    def replace_pair(self, index: str, pair: tuple[str]):
        new_tokens = []
        i = 0
        while i < len(self.tokens):
            if i < len(self.tokens) - 1 and (self.tokens[i], self.tokens[i+1]) == pair:
                new_tokens.append(index)
                i += 2
            else:
                new_tokens.append(self.tokens[i])
                i += 1
        self.tokens = new_tokens

    def bpe_compress(self):

        next_token_id = 0

        while True:
            counts = self.get_counts()
            if not counts:
                break
                
            max_pair = max(counts, key=counts.get)
            if counts[max_pair] <= 1:
                break

            test_codes = self.codes.copy()
            token_str = f'{next_token_id}'
            test_codes[token_str] = max_pair

            if not self.check_bytes(test_codes):
                break

            self.codes[token_str] = max_pair
            self.replace_pair(token_str, max_pair)
            next_token_id += 1
        
        return self.codes

In [125]:
test = BPEMethod('text.txt')
print(test.get_counts())

{('м', 'и'): 504, ('и', 'х'): 192, ('х', 'а'): 71, ('а', 'и'): 19, ('и', 'л'): 669, ('л', ' '): 811, (' ', 'б'): 692, ('б', 'у'): 115, ('у', 'л'): 257, ('л', 'г'): 14, ('г', 'а'): 199, ('а', 'к'): 479, ('к', 'о'): 1203, ('о', 'в'): 1337, ('в', '.'): 46, ('.', 'р'): 9, ('р', 'о'): 1115, ('о', 'к'): 341, ('в', 'ы'): 478, ('ы', 'е'): 207, ('е', ' '): 1557, (' ', 'я'): 201, ('я', 'й'): 46, ('й', 'ц'): 45, ('ц', 'а'): 119, ('а', 'г'): 126, ('г', 'л'): 234, ('л', 'а'): 736, ('а', 'в'): 315, ('в', 'а'): 784, ('а', ' '): 1581, (' ', '1'): 53, ('1', '.'): 2, ('.', 'к'): 22, ('к', 'у'): 299, ('у', 'р'): 174, ('р', 'р'): 12, ('р', 'и'): 602, ('и', 'к'): 506, ('л', 'ю'): 84, ('ю', 'м'): 4, ('м', ' '): 702, (' ', 'в'): 2020, ('в', 'и'): 458, ('и', 'т'): 407, ('т', 'э'): 2, ('э', ' '): 1, (' ', 'п'): 2336, ('п', 'р'): 849, ('о', 'ф'): 173, ('ф', 'е'): 172, ('е', 'с'): 650, ('с', 'с'): 231, ('с', 'о'): 486, ('о', 'р'): 780, ('р', 'а'): 1054, ('п', 'е'): 520, ('е', 'р'): 1136, ('р', 'с'): 283, ('с', '

In [126]:
print(test.bpe_compress())

{'0': (',', ' '), '1': (' ', 'п'), '2': ('о', ' '), '3': ('и', ' '), '4': ('а', ' '), '5': ('е', ' '), '6': ('о', 'в'), '7': ('с', 'т'), '8': ('а', 'л'), '9': ('е', 'р'), '10': ('р', 'о'), '11': ('р', 'а'), '12': (' ', 'в'), '13': ('е', 'н'), '14': (' ', 'н'), '15': ('к', 'о'), '16': ('е', 'л'), '17': ('к', 'а'), '18': ('т', 'о'), '19': (' ', 'с'), '20': ('н', 'о'), '21': ('1', 'о'), '22': ('и', 'л'), '23': ('е', 'т'), '24': ('а', 'н'), '25': ('й', ' '), '26': ('я', ' '), '27': ('н', 'ы'), '28': ('.', ' '), '29': ('е', 'с'), '30': ('з', 'а'), '31': ('н', 'а'), '32': ('е', 'м'), '33': ('о', 'т'), '34': ('о', 'л'), '35': ('р', 'и'), '36': ('т', '2'), '37': ('ь', ' '), '38': ('и', 'к'), '39': ('и', 'н'), '40': ('о', 'н'), '41': ('г', '2'), '42': ('л', 'а'), '43': ('и', 'т'), '44': ('м', ' '), '45': ('о', 'р'), '46': ('1', '9'), '47': ('о', 'д'), '48': ('у', ' '), '49': ('б', 'ы'), '50': ('0', 'п'), '51': ('в', ' '), '52': ('н', '2'), '53': ('и', 'з'), '54': ('а', 'т'), '55': ('в', 'ы'), '

In [96]:
t = '[1112]aaa[12]dd[1]dd ,. ddd'
print(re.findall(r'\[([0-9]+)\]|(.)', t))

[('1112', ''), ('', 'a'), ('', 'a'), ('', 'a'), ('12', ''), ('', 'd'), ('', 'd'), ('1', ''), ('', 'd'), ('', 'd'), ('', ' '), ('', ','), ('', '.'), ('', ' '), ('', 'd'), ('', 'd'), ('', 'd')]


In [129]:
print(''.join(test.tokens))

миха22 булгак6.10к6ы5яйцаг42в41.кур35кулюм1243э110ф29со1146с386а16 апр16264628 г47а0веч9ом5010ф29с45 зо34оги3iv г57удар7в132041унив9с432343ди61к18р зоо39743ут451м57кве46с38612ош161219во2517б392350омещающийся12 зоо39743уте0ч36н4улиц5г9ц13а.110ф29с45 30жег129хни25м546ы25шар 3огляд16ся.31ч82ужасающе2517та710фы14уж52сч435437з8ож132744им135251э18т зл57ча727й12еч9011в5217к46воп35чи20ю э182517та710фы19леду2319ч435437им1320110ф29с454в42димир4ип54ьевича46с386а.324849л2р65258 л2328г346430меч5416ь31я018л17чом0лысая0с1уч17м3ж16т654ых123457018рчащими12бо17м28лиц2г42дк255б3518е0нижня26губ455пяч134вп9ед2833 э18го46с386с155лиц2веч5220с222н4себ5н2915льк217п35з272533печа18к.144к11с20м1457487а10м47275м813ь595очк3в199ебря2025оп11ве0г42зк3бле7ящие0неб34ьшие0107у12ысо15го0сутул65428г6452219к35пучим018н59м0ква17ющи44г3457о443с61д3д56ги60711н207е25им16 такую 15гд4г64522 ч18-либ2в29к23ув913200у1730т16ь27й18ец111во255659161в11щ812 крючок 3щур22 г42з59284так 17к 40 г6452212сегд4ув913200иб2э56диция12 е41об4273у14